# Dataset Download Link
https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000

---
## Install Dependencies

In [1]:
# Uncomment if needed
# !pip install torch torchvision scikit-learn matplotlib pandas numpy pillow tqdm

import torch, torchvision, sklearn, matplotlib, pandas, numpy
print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"sklearn:     {sklearn.__version__}")
print(f"numpy:       {numpy.__version__}")

torch:       2.11.0+cu126
torchvision: 0.26.0+cu126
sklearn:     1.7.2
numpy:       2.2.6


---
## Global Configuration

In [2]:
import os, random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths — adjust DATA_DIR to your dataset location
BASE_DIR      = Path(".")
DATA_DIR      = BASE_DIR / "archive"
IMG_DIR_1     = DATA_DIR / "HAM10000_images_part_1"
IMG_DIR_2     = DATA_DIR / "HAM10000_images_part_2"
META_PATH     = DATA_DIR / "HAM10000_metadata.csv"
OUT_DIR       = BASE_DIR / "outputs"
PLOTS_DIR     = OUT_DIR / "plots"
MODELS_DIR    = OUT_DIR / "models"
SYNTHETIC_DIR = OUT_DIR / "synthetic_images"  # subfolders per class

for d in [OUT_DIR, PLOTS_DIR, MODELS_DIR, SYNTHETIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Model config
IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_CLASSES   = 7
EPOCHS_PHASE1 = 5
EPOCHS_PHASE2 = 15
LR_PHASE1     = 1e-3
LR_PHASE2     = 1e-4
DROPOUT_RATE  = 0.4
WEIGHT_DECAY  = 1e-4

# DCGAN config
Z_DIM         = 100
GAN_FEAT_G    = 64
GAN_FEAT_D    = 64
GAN_IMG_SIZE  = 64
GAN_EPOCHS    = 200
LR_GAN        = 2e-4
BETA1_GAN     = 0.5

# ── KEY FIX: target count for all classes ─────────────────────────────────
# HAM10000 majority class (nv) has 6705 images.
# We upsample EVERY class to this count using real + GAN synthetic images.
# This eliminates class imbalance completely.
TARGET_COUNT  = 6705

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# HAM10000 classes
CLASS_NAMES   = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS  = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_FULLNAME = {
    "akiec": "Actinic Keratosis",  "bcc":  "Basal Cell Carcinoma",
    "bkl":   "Benign Keratosis",   "df":   "Dermatofibroma",
    "mel":   "Melanoma",           "nv":   "Melanocytic Nevi",
    "vasc":  "Vascular Lesion",
}

print("\nClass index mapping:")
for k, v in CLASS_TO_IDX.items():
    print(f"  {v}: {k:6s} -> {CLASS_FULLNAME[k]}")
print(f"\nTarget count per class: {TARGET_COUNT:,} (majority class size)")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Class index mapping:
  0: akiec  -> Actinic Keratosis
  1: bcc    -> Basal Cell Carcinoma
  2: bkl    -> Benign Keratosis
  3: df     -> Dermatofibroma
  4: mel    -> Melanoma
  5: nv     -> Melanocytic Nevi
  6: vasc   -> Vascular Lesion

Target count per class: 6,705 (majority class size)


---
## Write config.py

In [3]:
config_txt = '''
import os, random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR      = Path(".")
DATA_DIR      = BASE_DIR / "archive"
IMG_DIR_1     = DATA_DIR / "HAM10000_images_part_1"
IMG_DIR_2     = DATA_DIR / "HAM10000_images_part_2"
META_PATH     = DATA_DIR / "HAM10000_metadata.csv"
OUT_DIR       = BASE_DIR / "outputs"
PLOTS_DIR     = OUT_DIR / "plots"
MODELS_DIR    = OUT_DIR / "models"
SYNTHETIC_DIR = OUT_DIR / "synthetic_images"
IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_CLASSES   = 7
EPOCHS_PHASE1 = 5
EPOCHS_PHASE2 = 15
LR_PHASE1     = 1e-3
LR_PHASE2     = 1e-4
DROPOUT_RATE  = 0.4
WEIGHT_DECAY  = 1e-4
Z_DIM         = 100
GAN_FEAT_G    = 64
GAN_FEAT_D    = 64
GAN_IMG_SIZE  = 64
GAN_EPOCHS    = 200
LR_GAN        = 2e-4
BETA1_GAN     = 0.5
TARGET_COUNT  = 6705
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES   = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS  = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_FULLNAME = {"akiec":"Actinic Keratosis","bcc":"Basal Cell Carcinoma","bkl":"Benign Keratosis","df":"Dermatofibroma","mel":"Melanoma","nv":"Melanocytic Nevi","vasc":"Vascular Lesion"}
'''

with open("config.py", "w") as f:
    f.write(config_txt)
print("config.py written ok")
print(f"Next -> run 01_eda.ipynb")

config.py written ok
Next -> run 01_eda.ipynb
